# Шаг 3. Загрузка файлов

Проверяем функции из `app/loader.py` на тестовых данных.

In [1]:
# Подготовка: чтобы ноутбук видел папку app и подхватывал изменения в коде без перезапуска
import sys

sys.path.append("..")

%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from app.config import SYNTHETIC_DIR
from app.loader import find_header_row, guess_separator, load, read_csv, read_excel

## 1. Читаем CSV кафе нашей функцией

In [3]:
cafe = read_csv(SYNTHETIC_DIR / "cafe.csv")
cafe.head()

,Дата чека,Позиция,Раздел меню,"Кол-во, шт","Выручка, руб"
0,01.10.24,Американо,Кофе,11.0,"55,00"
1,01.10.24,Бизнес-ланч,Обеды,26.0,"478,40"
2,01.10.24,Капучино,Кофе,22.0,"143,00"
3,01.10.24,Чизкейк,Десерты,6.0,"58,80"
4,02.10.24,Американо,Кофе,19.0,"95,00"


In [4]:
# Размер таблицы (строки, колонки) и типы колонок
print(cafe.shape)
cafe.dtypes

(2921, 5)


Дата чека           str
Позиция             str
Раздел меню         str
Кол-во, шт      float64
Выручка, руб        str
dtype: object

In [5]:
# Последние строки - видна строка "Итого", её уберём на шаге очистки
cafe.tail()

,Дата чека,Позиция,Раздел меню,"Кол-во, шт","Выручка, руб"
2916,30.09.26,Американо,Кофе,19.0,"95,00"
2917,30.09.26,Бизнес-ланч,Обеды,27.0,"496,80"
2918,30.09.26,Капучино,Кофе,27.0,"175,50"
2919,30.09.26,Чизкейк,Десерты,15.0,"147,00"
2920,Итого,NaN,NaN,NaN,"828 428,10"


## 2. Попробуй сломать: зачем нужна каждая часть функции

Запусти ячейки ниже и посмотри, что получается без нашей обработки.

In [6]:
# Без указания кодировки - pandas ждёт utf-8 и падает с ошибкой
pd.read_csv(SYNTHETIC_DIR / "cafe.csv")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc4 in position 0: invalid continuation byte

In [ ]:
# С правильной кодировкой, но без разделителя - всё в одной колонке
pd.read_csv(SYNTHETIC_DIR / "cafe.csv", encoding="cp1251").head()

In [ ]:
# Автоопределение разделителя pandas - путается из-за запятых в "Выручка, руб" и "55,00"
pd.read_csv(SYNTHETIC_DIR / "cafe.csv", encoding="cp1251", sep=None, engine="python").head()

In [ ]:
# А наша функция смотрит на заголовок и считает, чего больше - ; или ,
print(guess_separator("Дата чека;Позиция;Раздел меню;Кол-во, шт;Выручка, руб"))
print(guess_separator("date,product,qty,amount"))

## 3. Excel магазина: где начинается таблица

Над таблицей - шапка отчёта. Ищем строку, где начинаются настоящие названия колонок.

In [ ]:
# Обычное чтение - pandas принял первую строку "Отчёт о продажах" за заголовок
pd.read_excel(SYNTHETIC_DIR / "shop.xlsx").head()

In [ ]:
# Читаем "сырым" - без заголовка (header=None), чтобы увидеть все строки как есть
raw = pd.read_excel(SYNTHETIC_DIR / "shop.xlsx", header=None)
raw.head(6)

In [ ]:
# Сколько заполненных ячеек в каждой строке - именно на это смотрит find_header_row
raw.notna().sum(axis=1).head(6)

In [ ]:
# Наша функция: номер строки, где начинается таблица (считаем с нуля)
find_header_row(raw)

In [ ]:
# read_excel: все листы файла, у каждого найдена шапка. Результат - словарь {лист: таблица}
shop_sheets = read_excel(SYNTHETIC_DIR / "shop.xlsx")
print(list(shop_sheets))  # названия листов
shop = shop_sheets["Sheet1"]
shop.head()

## 4. load - одна функция для любого файла

Сама смотрит на расширение и вызывает нужную функцию. Всегда возвращает словарь {лист: таблица}.

In [ ]:
# Загружаем все три файла одной функцией
for file_name in ["shop.xlsx", "cafe.csv", "salon.xlsx"]:
    for sheet_name, table in load(SYNTHETIC_DIR / file_name).items():
        print(f"{file_name} / {sheet_name}: {table.shape[0]} строк, колонки: {list(table.columns)}")